# CNN Classification - Sklearn Digits

Convolutional Neural Network on the Sklearn Digits Dataset.

**Steps:** Load Data -> Preprocess -> Build -> Compile -> Train -> Evaluate -> Predict

In [ ]:
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
print('Libraries loaded')

## 1. Load Data from Folder

In [ ]:
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

X = np.load(os.path.join(DATA_DIR, 'X.npy'))
y = np.load(os.path.join(DATA_DIR, 'y.npy'))
target_names = np.load(os.path.join(DATA_DIR, 'target_names.npy'))

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Classes:', target_names)

## 2. Preprocess

In [ ]:
X_norm = X / 16.0
X_reshaped = X_norm.reshape(-1, 8, 8, 1).astype(np.float32)

lb = LabelBinarizer()
y_enc = lb.fit_transform(y)

X_train, X_temp, y_train, y_temp = train_test_split(X_reshaped, y_enc, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print('Train:', X_train.shape)
print('Val:  ', X_val.shape)
print('Test: ', X_test.shape)

## 3. Build CNN

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(8, 8, 1)),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    Conv2D(64, (2, 2), activation='relu', padding='same'),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(10, activation='softmax'),
])
model.summary()

## 4. Compile

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('Model compiled')

## 5. Train

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    verbose=1
)

## 6. Evaluate

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test Loss:     {loss:.4f}')

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=[str(c) for c in target_names]))

print('Confusion Matrix:')
print(confusion_matrix(y_true, y_pred))

## 7. Predict Single Sample

In [ ]:
idx = 5
sample = X_test[idx].reshape(1, 8, 8, 1)
true_label = int(np.argmax(y_test[idx]))

pred_prob = model.predict(sample, verbose=0)[0]
predicted = int(np.argmax(pred_prob))
confidence = pred_prob[predicted] * 100

print(f'True Label:  {true_label}')
print(f'Predicted:   {predicted}')
print(f'Confidence:  {confidence:.2f}%')
print(f'Status:      {"Correct" if predicted == true_label else "Incorrect"}')